# Segmentation experiments: field colors, whole-board outliers and metal cleanup

This compares the same image and rectification across nine variants. It does
not infer dart counts, tip positions or scores. Run the reference setup once,
inspect a selected image, then use the compact batch or manual-probe section.

**Color strategies:** `field` compares each pixel with its expected field
model; `board_union` uses the same fitted models but accepts any board color;
`global` fits four unlabeled robust color modes to the complete observed board
distribution, including wires, with deterministic K-means initialization.
Features are value-weighted HSV cone coordinates, so hue wraps continuously.

**Cleanup:** `none` retains raw anomalies; `hard` masks nominal grid bands and
opens small support; `support` retains non-wire interior components and restores
raw evidence only within 3 mm of them. No recursive grid regrowth or opening.

The default remains `field/support`: on the initial four sparsely annotated development
photos, whole-board models lose white flights/metal shafts that resemble board
paint. This is a small, tuned, same-board experiment—not a general benchmark.
Some background reflections and color-matching dart regions remain unresolved.
Two additional noisy challenge photos are included specifically to expose those failures.

## Kernel and rerunning after code changes

Select this repository's `.venv` Python kernel (created with
`uv sync --group dev`). After pulling code or changing stage/result
contracts, use **Restart Kernel**, then **Run All** from the top.
This also resolves stale imports such as `ColorAssignment` and old
analysis artifacts such as missing `color_model_names`. Rerunning
only a display cell cannot update an existing analysis result.
Autoreload picks up subsequent Python edits; rerun the
analysis cells to recompute results. Restart again for structural
class/dataclass changes rather than mixing old and new objects.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import sys
from functools import partial
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository or its examples directory.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from dartcv import (
    Printed20OrientationStage, extract_printed20_template,
    format_timing_summary, normalize_board_view, prepare_board,
)
from dartcv.color_dart_inspection import plot_orientation

IMAGE_DIR = ROOT / 'data/images/multi_dart'
MANIFEST = json.loads((ROOT / 'data/annotations/color_dart_orientation.json').read_text())
PHOTO_PATHS = sorted(IMAGE_DIR.glob('*.jpg'))
if not PHOTO_PATHS:
    raise FileNotFoundError('Restore data/images using the README DVC workflow first.')
NORMALIZER = partial(normalize_board_view, number_ring_radius_mm=240.0)

def read_photo(name):
    image = cv2.imread(str(IMAGE_DIR / name))
    if image is None:
        raise FileNotFoundError(IMAGE_DIR / name)
    return image

def source_label_angle(preparation, source_xy):
    point = np.linalg.inv(preparation.board.model_to_source) @ np.array([*source_xy, 1.0])
    point = point[:2] / point[2]
    return float(np.rad2deg(np.arctan2(point[0], -point[1])))


## Prepare the explicit reference template once

The manifest stores approximate manually inspected source-image label centers, not angles predicted by the matcher. Projection into the detected board model converts that independent observation to an angle. No source imagery is embedded in this notebook. The annular crop spans radii 202–228 mm and 14 degrees; inspect it before trusting the matcher.

In [ ]:
reference = MANIFEST['reference']
reference_preparation = prepare_board(
    read_photo(reference['image']), board_normalization=NORMALIZER,
)
if reference_preparation.failure is not None:
    raise RuntimeError(reference_preparation.failure)
reference_angle = source_label_angle(reference_preparation, reference['label_center_source_px'])
template = extract_printed20_template(
    reference_preparation.board_view, label_angle_deg=reference_angle,
)
ORIENTATION_STAGE = Printed20OrientationStage(template)
plt.figure(figsize=(4, 3))
plt.imshow(cv2.cvtColor(template, cv2.COLOR_BGR2RGB))
plt.title('Reference 20 — inspect this template')
plt.axis('off')
plt.show()
print(format_timing_summary(reference_preparation))


In [ ]:
from dartcv.color_dart_evaluation import (
    compare_segmentations, evaluate_sparse_probes,
    plot_segmentation_comparison, segmentation_variants,
)
from dartcv.color_dart_inspection import likelihood_overlay
from dartcv import BoardGeometry, OrientationEstimate
from dartcv.notebook_batch import BatchOutput, image_batch_figures, read_batch_image

IMAGE_INDEX = 5  # Deliberately noisy challenge case; try 0, 1, 7, 8 and 12 as well.
NORMALIZED_SIZE_PX = 768
image_path = PHOTO_PATHS[IMAGE_INDEX]
preparation = prepare_board(
    read_photo(image_path.name), board_normalization=NORMALIZER,
    normalized_size_px=NORMALIZED_SIZE_PX,
)
if preparation.failure is not None:
    raise RuntimeError(preparation.failure)
orientation = ORIENTATION_STAGE(preparation.board_view, diagnostics=False)
if orientation.value is None:
    raise RuntimeError(orientation.failure)
print(image_path.name, orientation.value)
print("If numbered orientation is weak, color fields retain detector paint alignment.")
variants = compare_segmentations(
    preparation.board_view, preparation.board, orientation.value
)
display(plot_segmentation_comparison(preparation.board_view, variants))

## Compact batch: selected strategy, final outputs only

Select `BATCH_VARIANT` from the nine names, choose files, then enable batching.
The loop retains only final thumbnails. Repeat with another variant to compare
the same inputs. Copy any bad filename into `IMAGE_INDEX` above for all nine
overlays, or investigate masks/models in the detailed segmentation notebook.

In [ ]:
import matplotlib.pyplot as plt

RUN_BATCH = False
BATCH_PATHS = PHOTO_PATHS[:3]
BATCH_VARIANT = "field/support"
batch_stage = segmentation_variants()[BATCH_VARIANT]

def batch_output(path):
    prepared = prepare_board(
        read_batch_image(path), board_normalization=NORMALIZER,
        normalized_size_px=NORMALIZED_SIZE_PX,
    )
    if prepared.failure is not None:
        return BatchOutput(None, f"FAILED: {prepared.failure.message}")
    oriented = ORIENTATION_STAGE(prepared.board_view, diagnostics=False)
    if oriented.value is None:
        return BatchOutput(None, f"Orientation FAILED: {oriented.failure}")
    outcome = batch_stage(
        prepared.board_view, prepared.board, oriented.value, diagnostics=False
    )
    if outcome.value is None:
        return BatchOutput(None, f"Segmentation FAILED: {outcome.failure}")
    status = "numbered orientation confident" if oriented.value.confident else "UNCERTAIN numbered orientation"
    return BatchOutput(
        likelihood_overlay(prepared.board_view.image, outcome.value.probabilities),
        f"{BATCH_VARIANT}; {status}; not confirmed darts",
    )

if RUN_BATCH:
    for batch_figure in image_batch_figures(BATCH_PATHS, batch_output, page_size=6):
        display(batch_figure)
        plt.close(batch_figure)

## Repeatable sparse-probe comparison

The manifest contains manually inspected flight, shaft, board and wire points
on four initial regression photos and two additional noisy challenge photos,
in explicitly stored reference rectifications. They are NOT
full dart masks. Their reference homographies and number-match estimates are
frozen to isolate segmentation changes from detector/matcher drift.
The same fixed frame is used for every variant, regardless of its result.

A positive is a patch mean >=0.5. Rates on flight/shaft probes measure retention;
rates on board/wire probes measure false positives. Report each numerator and
denominator: do not call these full-image precision, recall or IoU. Patches
outside observed scoring coverage are excluded and counted. Some flights are
outside the scoring image; that is a limitation, not a segmentation miss here.

`number_match` stores the old match outcome, NOT manual orientation truth.
One is deliberately wrong/uncertain to reproduce contamination of field models.
With the revised policy, its paint alignment remains detector-defined while
numbered orientation stays uncertain. There is no `5`/`1` number fallback yet.

First global fitting can include a lazy scikit-learn import; timings exclude
source decoding/detection but include fitting and cleanup. No new dependencies
or network downloads. Enable the next cell to rerun the six-photo evaluation.
Regression and challenge results are reported separately: the latter still fail
to separate only darts and prevent overclaiming success on the initial subset.

In [ ]:
RUN_PROBE_EVALUATION = False
if RUN_PROBE_EVALUATION:
    manifest = json.loads(
        (ROOT / "data/annotations/color_dart_segmentation.json").read_text()
    )
    totals = {}
    for annotation in manifest["images"]:
        board = BoardGeometry(np.array(annotation["model_to_source"], dtype=float), 0)
        prepared_view = normalize_board_view(
            read_photo(annotation["image"]), board, NORMALIZED_SIZE_PX,
            diagnostics=False,
        ).value
        recorded_orientation = OrientationEstimate(**annotation["number_match"])
        outcomes = compare_segmentations(prepared_view, board, recorded_orientation)
        probes = {name: annotation[name] for name in ("flight", "shaft", "board", "wire")}
        for name, outcome in outcomes.items():
            if outcome.value is None:
                raise RuntimeError(f"{annotation['image']} / {name}: {outcome.failure}")
            metrics = evaluate_sparse_probes(
                prepared_view, board, outcome.value.probabilities,
                np.array(annotation["model_to_source"], dtype=float), probes,
            )
            counts = totals.setdefault((annotation['role'], name), {category: [0, 0, 0] for category in probes})
            for category, metric in metrics.items():
                counts[category][0] += metric.positive
                counts[category][1] += metric.eligible
                counts[category][2] += metric.excluded
    print("variant | flight retained | shaft retained | board false positives | wire false positives")
    for (role, name), counts in totals.items():
        print(role, name, " | ".join(
            f"{positive}/{eligible} ({excluded} excluded)"
            for positive, eligible, excluded in counts.values()
        ))

## Evaluate manually annotated visible dart regions

Run `uv run python -m dartcv.annotate` from the repository root, open the
printed localhost URL, and annotate/complete the selected photograph.
See `docs/dart-annotation.md` for the labeling protocol. Enable the cell below.

**Dart retained** measures labeled dart pixels surviving segmentation.
**Detected pixels that are darts** measures how much foreground is real dart
rather than board/grid noise. Higher is better for both. Part retention
separates flights from thin shafts/barrels. IoU is a secondary overlap score.

Only completed visible polygons in observed scoring-board coverage are used.
Ignore regions and inferred silhouettes do not count as truth; tips are
saved for future localization evaluation. Coverage reports the fraction of
source-labeled dart pixels eligible here, so cropped/off-board darts remain
explicitly excluded. `n/a` means there are no applicable pixels, not zero.
These are per-photo development metrics, not held-out accuracy or instance
recognition. The error overlay shows retained darts, erased darts and noise.

In [ ]:
from dartcv.dart_annotations import (
    annotation_path, load_annotation, evaluate_visible_masks,
    plot_visible_mask_evaluation,
)

RUN_LABEL_EVALUATION = False
LABEL_DIR = ROOT / 'data/annotations/darts' / image_path.parent.name
LABEL_THRESHOLD = 0.5
LABEL_OVERLAY_VARIANT = 'field/support'

def percentage(value):
    return 'n/a' if value is None else f'{value:.1%}'

if RUN_LABEL_EVALUATION:
    label_file = annotation_path(LABEL_DIR, image_path.name)
    if not label_file.exists():
        print(f'No visible-region labels for {image_path.name}. Run the annotation tool first.')
    else:
        labels = load_annotation(label_file, source_image=image_path)
        if labels.status != 'completed':
            print(f'{image_path.name}: draft labels are excluded. Complete the photograph in the editor.')
        else:
            if LABEL_OVERLAY_VARIANT not in variants:
                raise ValueError(f'Choose an overlay variant from {list(variants)}')
            print('variant | dart retained | detected pixels that are darts | flight retained | shaft/barrel retained | IoU | label coverage')
            for name, outcome in variants.items():
                if outcome.value is None:
                    print(f'{name}: FAILED: {outcome.failure}')
                    continue
                evaluation = evaluate_visible_masks(
                    preparation.board_view, outcome.value.probabilities,
                    labels, threshold=LABEL_THRESHOLD,
                )
                metric = evaluation.metrics
                print(name, ' | '.join(percentage(value) for value in (
                    metric.recall, metric.precision,
                    metric.part_recall['flight'], metric.part_recall['shaft_barrel'],
                    metric.iou, metric.label_coverage,
                )))
                if name == LABEL_OVERLAY_VARIANT:
                    display(plot_visible_mask_evaluation(evaluation))